# Production-Hardened Digital Twin & AI Automation Business Assistant

**Author:** MD Ajharul Islam Akanda (Ajhar) — AI Automation Engineer, Full-Stack Backend Engineer & Technical Product Manager  
**Course Reference:** Week 1 Day 4 (`1_foundations/3_lab3.ipynb`) — Hand-Building an Autonomous Agent Loop & Multi-LLM Evaluator-Optimizer Pipeline

---

## 1. Executive Summary & Exercise Objectives

This project extends **Lab 3** into a production-ready **AI Automation Business Assistant & Portfolio Digital Twin** that fulfills both advanced course exercises:

1. **Exercise 1 — Multi-LLM Evaluator-Optimizer Guard (`strictly related to work only`):**
   - After the primary generator LLM forms its reply (and completes any tool calls), an independent **Evaluator LLM** audits the candidate reply against a strict **Pydantic v2 schema (`WorkEvaluationResult`)**.
   - Any leakage of personal trivia (such as food/cheese preferences in `summary.txt`), off-topic tangents, or ungrounded claims is rejected (`approved=False`), triggering a bounded **Optimizer Self-Correction Loop** (`MAX_EVAL_RETRIES = 2`) with a deterministic safe fallback.
2. **Exercise 2 — Business Domain Assistant & Deterministic Lead Capture:**
   - Ingests three structured knowledge sources: `twin/linkedin.pdf`, `twin/summary.txt`, and `twin/business_services.md` (detailing Ajhar's AI Automation & Backend Consultancy services across `LangGraph`, `n8n`, `Make.com`, `GoHighLevel`, `RAG`, `MCP`, `Python/FastAPI`, `Java/Spring Boot`, and `PostgreSQL`).
   - Equips the agent with two **Pydantic-validated tools**:
     - `record_business_lead`: Deterministically validates email syntax, deduplicates existing contacts, and persists leads to both structured `leads.jsonl` and human-readable `leads.txt`.
     - `record_unanswered_question`: Captures out-of-scope client or recruiter questions into `unanswered_questions.jsonl` so the assistant never hallucinates.

---

## 2. Architectural Gap Analysis: Baseline vs. Production Next Version

| Engineering Dimension | Baseline (`3_lab3.ipynb`) | Earlier Prototype | This Production Version (`ajhar_digital_twin`) |
| :--- | :--- | :--- | :--- |
| **Work-Only Guardrail** | Single LLM call; happily discusses personal food/cheese trivia from `summary.txt`. | Had an Evaluator call, but prompt explicitly *allowed* personal trivia, defeating the guard. | **Strict Zero-Tolerance Work Policy.** Evaluator rejects personal/non-work replies and forces an on-topic professional rewrite. |
| **Schema & Type Safety** | Raw strings & untyped dicts; crashed on `TypeError` string concatenation. | Raw `json.loads()` that failed open (`passed: True`) on parse errors. | **Pydantic v2 (`extra='forbid'`)** for tool inputs and evaluator outputs, plus deterministic fence stripping. |
| **Lead Capture Hygiene** | Appended raw string to `emails.txt`; no validation. | Recorded invalid/duplicate emails (`qqq@gmail.com`, duplicate `Sarah`). | **Regex + Pydantic Validation & Deduplication.** Updates existing leads cleanly and writes structured JSONL + text logs. |
| **Loop & API Resilience** | Unbounded `while` loop; hit 429 quota crash (`gemini-2.5-flash`). | Unbounded tool loop; no retry backoff. | **Bounded `MAX_TOOL_ROUNDS = 3` & `MAX_EVAL_RETRIES = 2`**, explicit timeouts, and exponential backoff on transient errors. |

## 3. Environment Setup & Resilient LLM Client

We configure the **OpenAI Python SDK** against Google's Gemini OpenAI-compatible endpoint using `gemini-3.1-flash-lite` (verified high free-tier quota and native tool-calling support), wrapped with exponential backoff for production resilience.

In [ ]:
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import APIError, APITimeoutError, OpenAI, RateLimitError
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator
from pypdf import PdfReader

# 1. Resolve environment variables (.env in current or parent workspace directories)
load_dotenv(override=True)
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    for parent in Path.cwd().parents:
        candidate_env = parent / ".env"
        if candidate_env.exists():
            load_dotenv(candidate_env, override=True)
            break

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL_NAME = "gemini-3.1-flash-lite"
API_TIMEOUT_SECONDS = 30.0
MAX_API_RETRIES = 3

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not google_api_key:
    raise ValueError("Missing GOOGLE_API_KEY or GEMINI_API_KEY in environment/.env!")

client = OpenAI(
    api_key=google_api_key,
    base_url=GEMINI_BASE_URL,
    timeout=API_TIMEOUT_SECONDS,
)

def call_llm_with_backoff(**kwargs: Any) -> Any:    #**kwargs means keyword arguement
    """
    Production wrapper around client.chat.completions.create with exponential backoff
    for transient RateLimitError (429) or API connection/timeout errors.
    """
    delay = 1.5
    for attempt in range(1, MAX_API_RETRIES + 1):
        try:
            return client.chat.completions.create(**kwargs)
        except (RateLimitError, APITimeoutError, APIError) as exc:
            if attempt == MAX_API_RETRIES:
                raise
            print(f"[RESILIENCE] Transient API error ({type(exc).__name__}). Retrying attempt {attempt}/{MAX_API_RETRIES} in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2.0

print(f"[INIT] Connected to Gemini endpoint using model '{MODEL_NAME}' (timeout={API_TIMEOUT_SECONDS}s)")

## 4. Multi-Source Business & Profile Context Ingestion

To make the assistant genuinely effective for **Exercise 2 (Business Assistant)**, we load:
1. `twin/linkedin.pdf` — Ajhar's verified career history, technical stack, and education.
2. `twin/summary.txt` — Executive overview (which also contains a personal food/cheese sentence used to test that our Evaluator Guard catches any personal drift).
3. `twin/business_services.md` — Concrete AI Automation & Backend Engineering service pillars, deliverables, and ROI metrics.

In [ ]:
BASE_DIR = Path.cwd()   # cwd() is current working directory
TWIN_DIR = BASE_DIR / "twin"

reader = PdfReader(str(TWIN_DIR / "linkedin.pdf"))
linkedin_pages = []
for idx, page in enumerate(reader.pages, start=1):
    extracted = page.extract_text()
    if extracted:
        linkedin_pages.append(f"--- LinkedIn Page {idx} ---\n{extracted.strip()}")
linkedin_context = "\n\n".join(linkedin_pages)

summary_context = (TWIN_DIR / "summary.txt").read_text(encoding="utf-8").strip()
business_context = (TWIN_DIR / "business_services.md").read_text(encoding="utf-8").strip()

print(f"[CONTEXT] LinkedIn PDF: {len(reader.pages)} pages ({len(linkedin_context)} chars)")
print(f"[CONTEXT] Summary text: {len(summary_context)} chars")
print(f"[CONTEXT] Business Services KB: {len(business_context)} chars")

## 5. Strict Pydantic v2 Schemas: Separating Deterministic Logic from LLM Outputs

In production AI engineering, we never trust raw LLM JSON strings or unvalidated user inputs. Below we define:
- `LeadCaptureInput`: Validates email format via regex, normalizes case, and forbids extra fields (`extra='forbid'`).
- `UnansweredQuestionInput`: Validates out-of-context business/technical questions for knowledge-base gap tracking.
- `WorkEvaluationResult`: Enforces a strict structured schema for the secondary **Evaluator LLM**.

In [ ]:
EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]{2,}$")

class LeadCaptureInput(BaseModel):
    """Strict Pydantic v2 schema for capturing prospective client or recruiter leads."""
    model_config = ConfigDict(extra="forbid")

    email: str = Field(..., description="The visitor's valid email address (e.g. sarah@agency.io)")
    name: str = Field(default="Not Provided", description="The visitor's name or company name")
    inquiry: str = Field(
        default="General AI Automation / Engineering Inquiry",
        description="Concise summary of the business automation need, consulting project, or role"
    )

    @field_validator("email")
    @classmethod
    def validate_and_normalize_email(cls, value: str) -> str:
        cleaned = value.strip().lower()
        if not EMAIL_REGEX.match(cleaned):
            raise ValueError(f"Invalid email address format: '{value}'. Must be a valid user@domain.tld address.")
        return cleaned

    @field_validator("name", "inquiry")
    @classmethod
    def sanitize_text_fields(cls, value: str) -> str:
        cleaned = " ".join(value.strip().split())
        return cleaned if cleaned else "Not Provided"


class UnansweredQuestionInput(BaseModel):
    """Strict Pydantic v2 schema for logging unanswered work/business questions."""
    model_config = ConfigDict(extra="forbid")

    # ... means this parameter is strictly required
    question: str = Field(..., min_length=3, description="The specific question that could not be answered from verified context")
    category: str = Field(default="out_of_context", description="Category of gap: pricing, availability, unsupported_tech, or other")


class WorkEvaluationResult(BaseModel):
    """Strict Pydantic v2 schema returned by the secondary Evaluator LLM."""
    model_config = ConfigDict(extra="forbid")

    is_strictly_work_related: bool = Field(
        ..., 
        description="True ONLY if the response is strictly professional and focused on Ajhar's work, engineering skills, career, or business services. False if it discusses personal hobbies, food/cheese preferences, recipes, politics, or non-work trivia."
    )
    is_grounded_and_accurate: bool = Field(
        ..., 
        description="True if the response is faithful to the provided profile/business context and does not hallucinate fake credentials."
    )
    approved: bool = Field(
        ..., 
        description="True ONLY if both is_strictly_work_related and is_grounded_and_accurate are True."
    )
    reasoning: str = Field(..., description="Concise justification for the pass/fail verdict.")
    critique: str = Field(
        default="",
        description="If approved is False, concrete instructions telling the primary assistant how to rewrite the reply so it is strictly work-related and professional."
    )

print("[SCHEMAS] Pydantic v2 schemas compiled with extra='forbid'.")

## 6. Deterministic Business Tools & Tool Registry

We implement two deterministic tools and register them in a dispatch dictionary (`TOOL_REGISTRY`):
1. `record_business_lead`: Validates arguments with `LeadCaptureInput`, checks `leads.jsonl` for duplicate email addresses (updating the existing record if found or appending a new record), and mirrors human-readable entries to `leads.txt`.
2. `record_unanswered_question`: Logs unverified or out-of-scope questions to `unanswered_questions.jsonl` so Ajhar can follow up accurately.

In [ ]:
LEADS_JSONL_PATH = BASE_DIR / "leads.jsonl"
LEADS_TXT_PATH = BASE_DIR / "leads.txt"
UNANSWERED_JSONL_PATH = BASE_DIR / "unanswered_questions.jsonl"

def _sync_leads_txt(records: list[dict[str, Any]]) -> None:
    """Writes a clean, human-readable summary of all deduplicated leads to leads.txt."""
    lines = [
        f"[{r['timestamp']}] Name: {r['name']} | Email: {r['email']} | Inquiry: {r['inquiry']} (Updates: {r.get('update_count', 1)})"
        for r in records
    ]
    LEADS_TXT_PATH.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

def record_business_lead(**raw_kwargs: Any) -> dict[str, Any]:
    """
    Validates and persists a business lead with email deduplication.
    """
    try:
        validated = LeadCaptureInput(**raw_kwargs)
    except ValidationError as exc:
        error_msg = exc.errors()[0].get("msg", "Invalid lead parameters")
        print(f"[TOOL VALIDATION FAILED] record_business_lead: {error_msg}")
        return {
            "status": "validation_error",
            "message": f"Could not record email due to validation error: {error_msg}. Please ask the user for a valid email address."
        }

    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    existing_records: list[dict[str, Any]] = []
    if LEADS_JSONL_PATH.exists():
        for line in LEADS_JSONL_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                existing_records.append(json.loads(line))

    # Check for existing email to deduplicate
    matched_record = next((r for r in existing_records if r["email"] == validated.email), None)
    if matched_record:
        matched_record["timestamp"] = timestamp
        if validated.name != "Not Provided":
            matched_record["name"] = validated.name
        if validated.inquiry != "General AI Automation / Engineering Inquiry":
            matched_record["inquiry"] = validated.inquiry
        matched_record["update_count"] = matched_record.get("update_count", 1) + 1
        action = "updated_existing_lead"
    else:
        new_record = {
            "lead_id": str(uuid.uuid4())[:8],
            "timestamp": timestamp,
            "name": validated.name,
            "email": validated.email,
            "inquiry": validated.inquiry,
            "update_count": 1,
        }
        existing_records.append(new_record)
        action = "created_new_lead"

    # Persist JSONL and human-readable TXT
    LEADS_JSONL_PATH.write_text(
        "\n".join(json.dumps(r) for r in existing_records) + "\n",
        encoding="utf-8"
    )
    _sync_leads_txt(existing_records)

    print(f"[TOOL EXECUTION] record_business_lead ({action}) -> Email: {validated.email} | Name: {validated.name} | Inquiry: {validated.inquiry}")
    return {
        "status": "success",
        "action": action,
        "email": validated.email,
        "name": validated.name,
        "message": f"Successfully recorded lead for {validated.email} ({action}). Ajhar will follow up shortly."
    }


def record_unanswered_question(**raw_kwargs: Any) -> dict[str, Any]:
    """
    Validates and logs an unanswered question to unanswered_questions.jsonl.
    """
    try:
        validated = UnansweredQuestionInput(**raw_kwargs)
    except ValidationError as exc:
        return {"status": "validation_error", "message": str(exc)}

    entry = {
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "question": validated.question,
        "category": validated.category,
    }
    with open(UNANSWERED_JSONL_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

    print(f"[TOOL EXECUTION] record_unanswered_question -> Category: {validated.category} | Question: '{validated.question}'")
    return {
        "status": "logged",
        "message": "Question logged for Ajhar's direct review. Inform the visitor honestly that you do not have that specific detail and offer to connect them via email."
    }


# Registry mapping tool names to deterministic functions
TOOL_REGISTRY: dict[str, Callable[..., dict[str, Any]]] = {
    "record_business_lead": record_business_lead,
    "record_unanswered_question": record_unanswered_question,
}

# OpenAI-compatible Tool Specifications
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "record_business_lead",
            "description": (
                "Record a prospective client's or recruiter's email address, name, and business inquiry "
                "when they want to get in touch, request an AI automation consultation, or discuss a role. "
                "Only call this tool when the user has explicitly provided an email address."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "The visitor's email address (e.g. client@company.com)"
                    },
                    "name": {
                        "type": "string",
                        "description": "The visitor's name or organization if provided"
                    },
                    "inquiry": {
                        "type": "string",
                        "description": "Summary of their business automation project, support need, or job opportunity"
                    }
                },
                "required": ["email"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_unanswered_question",
            "description": (
                "Always call this tool when a visitor asks a professional or business question whose answer "
                "is not present in Ajhar's LinkedIn profile or business services knowledge base."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The unanswered question asked by the visitor"
                    },
                    "category": {
                        "type": "string",
                        "description": "Category of the gap (e.g. pricing, availability, unsupported_service, other)"
                    }
                },
                "required": ["question"],
                "additionalProperties": False
            }
        }
    }
]

print(f"[TOOLS] Registered {len(TOOL_REGISTRY)} validated business tools: {list(TOOL_REGISTRY.keys())}")

## 7. System Prompts: Primary Business Assistant & Strict Work-Only Evaluator

Notice how we design **two distinct system prompts**:
1. `PRIMARY_SYSTEM_PROMPT`: Instructs the Digital Twin to act as Ajhar's AI Automation & Backend Engineering Business Assistant, grounded in `business_context`, `linkedin_context`, and `summary_context`.
2. `EVALUATOR_SYSTEM_PROMPT`: Enforces a **strict work-only policy**. Even though `summary_context` mentions personal food/cheese preferences, the Evaluator is explicitly programmed to **FAIL (`approved: false`)** any response that engages in personal food/hobby trivia, recipes, politics, or non-work topics!

In [ ]:
PRIMARY_SYSTEM_PROMPT = f"""# Role & Identity
You are the official AI Digital Twin and Business Consultancy Assistant for **MD Ajharul Islam Akanda (Ajhar)** — AI Automation Engineer, Full-Stack Backend Engineer, and Technical Product Manager.
You engage with prospective business clients, agency owners, engineering leaders, and technical recruiters visiting Ajhar's website.

# Verified Business Consultancy Knowledge Base
{business_context}

# Verified Executive Summary
{summary_context}

# Verified LinkedIn Career & Education Profile
{linkedin_context}

# Mandatory Operating Rules
1. **Strictly Work & Business Focused:** You ONLY discuss Ajhar's professional services, AI automation architectures (LangGraph, n8n, Make.com, GoHighLevel, RAG, MCP), backend engineering (Python/FastAPI, Java/Spring Boot, PostgreSQL, PL/SQL), career track record, and consulting engagements.
2. **Refuse Non-Work & Personal Trivia:** Never discuss personal food preferences (like cheese, pizza, French food), recipes, entertainment, politics, or general non-business topics — even if a personal note appears in the raw summary text. If asked about personal or off-topic subjects, politely decline and pivot immediately to how Ajhar helps businesses automate workflows and build backend systems.
3. **No Hallucinations & Gap Logging:** Rely exclusively on the verified context above. If a visitor asks a specific business or career question not covered in the context, call `record_unanswered_question` first, state transparently that you don't have that exact detail on hand, and invite them to leave their email.
4. **Proactive Lead Capture:** When a visitor wants to hire Ajhar, book a consultation, automate their business, or discuss a role, ask for their email address. Once they provide an email address, immediately call `record_business_lead`.
"""

EVALUATOR_SYSTEM_PROMPT = """You are an independent Corporate Compliance & Work-Relevance Evaluator LLM.
Your job is to audit a candidate reply generated by Ajhar's AI Business Assistant before it is shown to a website visitor.

### STRICT PASS CRITERIA (`approved: true`)
Both `is_strictly_work_related` and `is_grounded_and_accurate` MUST be `true`:
1. The response focuses strictly on professional work, AI automation services (n8n, Make, GHL, LangGraph, RAG, MCP), backend software engineering (FastAPI, Spring Boot, PostgreSQL, Fintech), career experience, education, or professional contact/lead capture.
2. Standard professional greetings, polite refusals of off-topic questions, and pivots back to Ajhar's business offerings ARE work-related and MUST PASS.

### STRICT FAIL CRITERIA (`approved: false`)
Mark `is_strictly_work_related: false` and `approved: false` if ANY of the following occur:
1. The assistant discusses personal hobbies, food preferences (e.g., French food, cheese, mozzarella, cream cheese, pizza, cheesecake), cooking recipes, sports, politics, or general trivia.
2. The assistant acts as a general-purpose chatbot solving unrelated homework or non-business tasks instead of representing Ajhar's consultancy.
3. The assistant fabricates credentials or makes ungrounded promises (`is_grounded_and_accurate: false`).

Return ONLY a valid JSON object matching this exact schema:
{
  "is_strictly_work_related": true,
  "is_grounded_and_accurate": true,
  "approved": true,
  "reasoning": "Brief explanation of why the response passed or failed.",
  "critique": "If approved is false, provide clear instructions to strip all personal/off-topic content and pivot strictly to Ajhar's AI automation and backend engineering work. Empty string if approved is true."
}
"""

print("[PROMPTS] Primary Business Assistant and Strict Work-Only Evaluator prompts ready.")

## 8. Bounded Autonomous Agent Loop & Multi-LLM Evaluator-Optimizer Pipeline

Here we build the core engine in three clean, modular functions:
1. `run_generator_agent(messages)`: Runs the primary LLM with tool calling enabled inside a bounded loop (`MAX_TOOL_ROUNDS = 3`), dispatching requested tools through `TOOL_REGISTRY`.
2. `evaluate_candidate_reply(user_query, candidate_reply)`: Invokes the secondary **Evaluator LLM** (`temperature=0.0`, `response_format={"type": "json_object"}`) and validates the output with Pydantic's `WorkEvaluationResult`.
3. `guarded_business_chat(message, history)`: Orchestrates the complete **Generator -> Evaluator -> Optimizer (Self-Correction)** workflow.

In [ ]:
MAX_TOOL_ROUNDS = 3
MAX_EVAL_RETRIES = 2

SAFE_WORK_FALLBACK = (
    "I am Ajhar's AI Digital Twin, dedicated strictly to discussing his AI Automation & Backend Engineering "
    "consultancy, technical architecture (LangGraph, n8n, FastAPI, Spring Boot), and career experience. "
    "How can I help you automate your business workflows or evaluate Ajhar for an engineering project? "
    "If you'd like to connect directly, please share your email address!"
)

def sanitize_gradio_history(history: list[Any]) -> list[dict[str, str]]:
    """Normalizes Gradio chat history into clean OpenAI-compatible role/content dicts."""
    cleaned: list[dict[str, str]] = []
    if not history:
        return cleaned
    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            content_val = item["content"]
            if isinstance(content_val, list):
                content_val = " ".join(str(part.get("text", part)) if isinstance(part, dict) else str(part) for part in content_val)
            cleaned.append({"role": str(item["role"]), "content": str(content_val)})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0] is not None:
                cleaned.append({"role": "user", "content": str(item[0])})
            if item[1] is not None:
                cleaned.append({"role": "assistant", "content": str(item[1])})
    return cleaned


def execute_tool_calls(tool_calls: list[Any]) -> list[dict[str, str]]:
    """Dispatches LLM tool calls through TOOL_REGISTRY with deterministic error handling."""
    tool_Messages: list[dict[str, str]] = []
    for tool_call in tool_calls:
        fn_name = tool_call.function.name
        try:
            raw_args = json.loads(tool_call.function.arguments or "{}")
        except json.JSONDecodeError:
            raw_args = {}

        handler = TOOL_REGISTRY.get(fn_name)
        if handler:
            result_payload = handler(**raw_args)
        else:
            result_payload = {"status": "error", "message": f"Unknown tool '{fn_name}'"}

        tool_Messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result_payload),
        })
    return tool_Messages


def run_generator_agent(messages: list[Any]) -> str:
    """
    Executes the primary Generator LLM with a bounded tool-calling loop (MAX_TOOL_ROUNDS).
    """
    response = call_llm_with_backoff(
        model=MODEL_NAME,
        messages=messages,
        tools=TOOLS_SCHEMA,
        temperature=0.3,
    )

    rounds = 0
    while response.choices[0].finish_reason == "tool_calls" and rounds < MAX_TOOL_ROUNDS:
        rounds += 1
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)

        tool_results = execute_tool_calls(assistant_msg.tool_calls or [])
        messages.extend(tool_results)

        response = call_llm_with_backoff(
            model=MODEL_NAME,
            messages=messages,
            tools=TOOLS_SCHEMA,
            temperature=0.3,
        )

    return (response.choices[0].message.content or "").strip()


def _extract_clean_json(raw_text: str) -> str:
    """Strips optional markdown code fences so Pydantic JSON parsing never trips on formatting."""
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def evaluate_candidate_reply(user_query: str, candidate_reply: str) -> WorkEvaluationResult:
    """
    Exercise 1: Calls an independent Evaluator LLM to verify that the candidate reply
    is strictly related to work/business and grounded in Ajhar's profile.
    """
    eval_messages = [
        {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"### Visitor Message:\n{user_query}\n\n"
                f"### Proposed Assistant Reply:\n{candidate_reply}\n\n"
                "Evaluate whether the Proposed Assistant Reply is strictly related to work/business and grounded. Return JSON only."
            ),
        },
    ]

    eval_response = call_llm_with_backoff(
        model=MODEL_NAME,
        messages=eval_messages,
        response_format={"type": "json_object"},
        temperature=0.0,
    )

    raw_content = _extract_clean_json(eval_response.choices[0].message.content or "{}")
    try:
        parsed_dict = json.loads(raw_content)
        result = WorkEvaluationResult(**parsed_dict)
        # Enforce invariant: approved can only be True if both sub-flags are True
        if not (result.is_strictly_work_related and result.is_grounded_and_accurate):
            result.approved = False
        return result
    except (json.JSONDecodeError, ValidationError) as exc:
        print(f"[EVALUATOR PARSE WARNING] Could not validate Evaluator JSON ({exc}). Failing closed for revision.")
        return WorkEvaluationResult(
            is_strictly_work_related=False,
            is_grounded_and_accurate=False,
            approved=False,
            reasoning="Evaluator schema validation failed; triggering safe work-focused regeneration.",
            critique="Rewrite the response so it is strictly professional and focuses only on Ajhar's AI automation and backend engineering services.",
        )


def guarded_business_chat(message: str, history: list[Any] | None = None) -> str:
    """
    Full Multi-LLM Agent Pipeline:
    1. Primary LLM generates answer (and invokes tools if needed).
    2. Secondary Evaluator LLM audits that the reply is strictly related to work.
    3. If rejected, Optimizer loop feeds the critique back for self-correction (up to MAX_EVAL_RETRIES).
    """
    correlation_id = str(uuid.uuid4())[:8]
    clean_history = sanitize_gradio_history(history or [])
    messages: list[Any] = [
        {"role": "system", "content": PRIMARY_SYSTEM_PROMPT},
        *clean_history,
        {"role": "user", "content": message},
    ]

    # Step 1: Primary Generator + Tool Execution Loop
    candidate_reply = run_generator_agent(messages)

    # Step 2: Multi-LLM Evaluator-Optimizer Loop
    for attempt in range(1, MAX_EVAL_RETRIES + 2):
        evaluation = evaluate_candidate_reply(user_query=message, candidate_reply=candidate_reply)
        print(
            f"[EVALUATOR AUDIT | trace={correlation_id} | attempt={attempt}] "
            f"Approved={evaluation.approved} (WorkRelated={evaluation.is_strictly_work_related}, Grounded={evaluation.is_grounded_and_accurate}) "
            f"| Reason: {evaluation.reasoning}"
        )

        if evaluation.approved:
            return candidate_reply

        if attempt <= MAX_EVAL_RETRIES:
            print(f"[OPTIMIZER SELF-CORRECTION | trace={correlation_id}] Rejecting reply & regenerating with critique: {evaluation.critique}")
            messages.append({"role": "assistant", "content": candidate_reply})
            messages.append({
                "role": "user",
                "content": (
                    f"[COMPLIANCE EVALUATOR REJECTION]: Your previous response was blocked because it violated our strict work-only policy. "
                    f"Evaluator Critique: {evaluation.critique or evaluation.reasoning}. "
                    "Rewrite your response immediately: remove ALL personal/off-topic details and strictly discuss Ajhar's professional AI automation and backend engineering work."
                ),
            })
            candidate_reply = run_generator_agent(messages)

    print(f"[FALLBACK TRIGGERED | trace={correlation_id}] Max evaluation retries reached. Returning deterministic work-only fallback.")
    return SAFE_WORK_FALLBACK

print("[PIPELINE] Multi-LLM Evaluator-Optimizer pipeline ready.")

## 9. Production Verification Suite (4 Automated Tests)

Let's verify every requirement of the upgraded architecture:
- **Test 1 (Business Architecture Consultation):** Visitor asks how Ajhar can automate their B2B agency's lead qualification and customer support using `n8n`, `LangGraph`, and `FastAPI`.
- **Test 2 (Adversarial Work-Only Evaluator Proof):** We test two levels:
  - **2A (Direct Evaluator Rejection & Optimizer Self-Correction Proof):** We simulate a flawed draft reply that leaks Ajhar's personal cheese/pizza food trivia from `summary.txt` and verify that `evaluate_candidate_reply()` flags `approved=False` and our optimizer rewrites it into a 100% professional response.
  - **2B (End-to-End Off-Topic Prompt):** Visitor asks about Ajhar's favorite cheese and pizza recipe.
- **Test 3 (Validated Lead Capture & Deduplication):** Visitor provides their contact email (`sarah.connor@apexautomation.io`) to book a discovery call, plus a second update with the same email to prove deduplication.
- **Test 4 (Unanswered Question Tool Logging):** Visitor asks an out-of-scope question (*"Do you offer on-site mainframe hardware repair in Zurich, and what is your hourly CHF rate?"*) to verify `record_unanswered_question` triggers without hallucinating.

In [ ]:
# Clean previous test artifacts for a fresh, deterministic verification run
for artifact_path in (LEADS_JSONL_PATH, LEADS_TXT_PATH, UNANSWERED_JSONL_PATH):
    if artifact_path.exists():
        artifact_path.unlink()

print("=" * 80)
print("TEST 1: Business Service & AI Automation Architecture Inquiry")
print("=" * 80)
reply_1 = guarded_business_chat(
    "We run a B2B service agency losing leads due to slow follow-ups and overwhelmed support reps. "
    "How can Ajhar help us automate our CRM pipeline and customer support, and what ROI has he delivered before?"
)
display(Markdown(reply_1))

In [ ]:
print("=" * 80)
print("TEST 2A: Adversarial Proof — Evaluator Catching & Rejecting Personal Trivia Leakage")
print("=" * 80)
flawed_non_work_draft = (
    "Ajhar loves French food! Strangely, he hates almost all forms of cheese except mozzarella "
    "and cream cheese because pizza and cheesecake are the greatest! Here is a great pizza recipe..."
)
audit_verdict = evaluate_candidate_reply(
    user_query="What kind of food and cheese does Ajhar like? Give me a pizza recipe.",
    candidate_reply=flawed_non_work_draft,
)
print(f"[DIRECT EVALUATOR VERDICT] Approved: {audit_verdict.approved} | Work-Related: {audit_verdict.is_strictly_work_related}")
print(f"[DIRECT EVALUATOR REASON]  {audit_verdict.reasoning}")
print(f"[DIRECT EVALUATOR CRITIQUE] {audit_verdict.critique}\n")
assert audit_verdict.approved is False, "Evaluator MUST reject personal food/cheese trivia!"

print("=" * 80)
print("TEST 2B: End-to-End Guarded Chat on Off-Topic Personal Question")
print("=" * 80)
reply_2 = guarded_business_chat(
    "What is Ajhar's favorite food and cheese, and can you share a quick homemade pizza recipe?"
)
display(Markdown(reply_2))

In [ ]:
print("=" * 80)
print("TEST 3: Validated Business Lead Capture & Email Deduplication")
print("=" * 80)
reply_3a = guarded_business_chat(
    "We'd love to hire Ajhar to build a LangGraph + n8n lead qualification agent for our agency. "
    "My name is Sarah Jenkins and my email is Sarah.Jenkins@ApexAgency.io."
)
display(Markdown(reply_3a))

print("\n--- Sending a follow-up from the SAME email to test deduplication ---")
reply_3b = guarded_business_chat(
    "Hi again, it's Sarah Jenkins (sarah.jenkins@apexagency.io). Please update my inquiry to note that we also need FastAPI & PostgreSQL RAG integration."
)
display(Markdown(reply_3b))

print("\n--- Verifying Persisted leads.txt ---")
print(LEADS_TXT_PATH.read_text(encoding="utf-8"))
print("--- Verifying Persisted leads.jsonl ---")
print(LEADS_JSONL_PATH.read_text(encoding="utf-8"))

In [ ]:
print("=" * 80)
print("TEST 4: Out-of-Context Question -> Unanswered Question Tool Trigger")
print("=" * 80)
reply_4 = guarded_business_chat(
    "Does Ajhar provide on-site physical mainframe soldering in Zurich, Switzerland, and what is his fixed daily rate in Swiss Francs (CHF)?"
)
display(Markdown(reply_4))

print("\n--- Verifying Persisted unanswered_questions.jsonl ---")
if UNANSWERED_JSONL_PATH.exists():
    print(UNANSWERED_JSONL_PATH.read_text(encoding="utf-8"))
else:
    print("No unanswered_questions.jsonl file found.")

## 10. Interactive Gradio Business Assistant UI

Run the cell below to launch the interactive **Gradio ChatInterface** in your browser (`inbrowser=True`).

In [ ]:
demo = gr.ChatInterface(
    fn=guarded_business_chat,
    title="🤖 Ajhar's AI Automation & Backend Consultancy Digital Twin",
    description=(
        "Welcome! I am the production Digital Twin of **MD Ajharul Islam Akanda (Ajhar)** — "
        "AI Automation Engineer, Full-Stack Backend Engineer & Technical Product Manager.\n\n"
        "**Guarded by a Multi-LLM Evaluator-Optimizer Pipeline:** Every response is audited in real time by a secondary Evaluator LLM "
        "to ensure strictly work-related, grounded technical consultation. Leave your email anytime to schedule a discovery call!"
    ),
    examples=[
        "How can you automate our agency's lead qualification and CRM workflows using n8n and GoHighLevel?",
        "Tell me about your production RAG and LangGraph agent architecture and the 75% support response time reduction.",
        "What is your enterprise backend experience with Java/Spring Boot, Python/FastAPI, and fintech payment systems?",
        "I'd like to hire Ajhar for a custom AI automation project. My name is Alex Rivera and my email is alex@riveraventures.com."
    ],
)

if __name__ == "__main__":
    # Set prevent_thread_lock=True during automated notebook execution so cells complete cleanly;
    # In interactive Jupyter usage, this still renders the inline iframe and opens the local server.
    demo.launch(inbrowser=False, prevent_thread_lock=True)